# GitHub Audit — Extension: Dual Account + Machine Attribution

**Run AFTER `github_audit.ipynb`** — this notebook adds:

| Section | Content |
|---------|---------|
| A | Dual-token setup (`skovnats` personal + `Opa-org` org) |
| B | SSH & GPG key inventory — fingerprints for both accounts |
| C | OAuth apps & authorized integrations |
| D | Enhanced commit forensics — timezone offset, signing key, session clustering |
| E | Machine attribution heuristics |
| F | Cross-account merged timeline |
| G | Limitations statement (court documentation) |

---
### The analogy
GitHub doesn't have a camera recording *which SSH key* authenticated each push — that log doesn't
exist in the public API. But every commit carries its own fingerprints: timezone offset, git identity,
and (if signed) the signing key. Like handwriting analysis, we combine signals to infer origin.

## Section A — Dual Account Setup

In [2]:
import os, json, base64, hashlib, re, time, warnings, requests
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from github import Github, GithubException
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ─── CONFIGURE ────────────────────────────────────────────────────────────────
GITHUB_TOKEN_PERSONAL = os.getenv("GITHUB_TOKEN")      or "ghp_YOUR_PERSONAL_TOKEN"
GITHUB_TOKEN_OPA      = os.getenv("GITHUB_TOKEN_Opa")  or "ghp_YOUR_OPA_TOKEN"

PERSONAL_LOGIN = "skovnats"
ORG_LOGIN      = "Opa-org"   # adjust if org name differs

OUTPUT_DIR = Path("github_audit")
EXT_DIR    = OUTPUT_DIR / "extension"
EXT_DIR.mkdir(parents=True, exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────

g_personal = Github(GITHUB_TOKEN_PERSONAL, per_page=100)
g_opa      = Github(GITHUB_TOKEN_OPA,      per_page=100)

user_personal = g_personal.get_user()
user_opa      = g_opa.get_user()

try:
    org_opa    = g_opa.get_organization(ORG_LOGIN)
    opa_is_org = True
except GithubException:
    org_opa    = g_opa.get_user(ORG_LOGIN)
    opa_is_org = False

rl_p = g_personal.get_rate_limit()
rl_o = g_opa.get_rate_limit()

print(f"{'Account':20s} {'Token':20s} {'Type':8s} Rate limit")
print("-" * 65)
print(f"{user_personal.login:20s} GITHUB_TOKEN         {'User':8s} {rl_p.core.remaining}/{rl_p.core.limit}")
print(f"{user_opa.login:20s} GITHUB_TOKEN_Opa     {'Org' if opa_is_org else 'User':8s} {rl_o.core.remaining}/{rl_o.core.limit}")

PERSONAL_TAG = "skovnats"
OPA_TAG      = "Opa-org"

def ts(dt) -> str:
    return dt.isoformat() if hasattr(dt, 'isoformat') else str(dt) if dt else ""

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)

def load_json(path):
    try:
        with open(path) as f: return json.load(f)
    except Exception: return []

def rate_limit_wait(client, buffer=100):
    remaining, limit = client.rate_limiting
    if remaining < buffer:
        reset_dt = datetime.fromtimestamp(client.rate_limiting_resettime, tz=timezone.utc)
        secs = (reset_dt - datetime.now(timezone.utc)).total_seconds() + 5
        print(f"  ⏳ Rate limit low ({remaining}/{limit}). Sleeping {secs:.0f}s …")
        time.sleep(max(secs, 0))

print("\n✅ Dual-client setup complete.")


Account              Token                Type     Rate limit
-----------------------------------------------------------------


AttributeError: 'RateLimitOverview' object has no attribute 'core'

## Section B — SSH & GPG Key Inventory

Lists every key on both accounts with its **fingerprint** — the stable ID you match against local keys.

To find your local key fingerprints run:
```bash
ssh-keygen -l -E sha256 -f ~/.ssh/id_ed25519.pub
ssh-keygen -l -E sha256 -f ~/.ssh/id_rsa.pub
# repeat for each key in ~/.ssh/
```

In [1]:
def compute_ssh_fingerprint(public_key_str: str) -> str:
    """
    SHA256 fingerprint matching `ssh-keygen -l -E sha256 -f key.pub`.
    Identifies the key without exposing the private key.
    """
    try:
        parts = public_key_str.strip().split()
        if len(parts) < 2:
            return "INVALID_FORMAT"
        key_bytes = base64.b64decode(parts[1])
        digest = hashlib.sha256(key_bytes).digest()
        return "SHA256:" + base64.b64encode(digest).decode('ascii').rstrip('=')
    except Exception as e:
        return f"ERROR:{e}"


def fetch_ssh_keys(gh_user, account_tag: str, token: str) -> list:
    records = []
    try:
        for key in gh_user.get_keys():
            raw_key = key.raw_data.get('key', '')
            fp = compute_ssh_fingerprint(raw_key)
            records.append({
                "account":     account_tag,
                "key_id":      key.id,
                "title":       key.title,       # label you gave when adding to GitHub
                "key_type":    raw_key.split()[0] if raw_key else '',
                "fingerprint": fp,              # match vs local ssh-keygen output
                "created_at":  key.raw_data.get('created_at', ''),
                "read_only":   key.raw_data.get('read_only', False),
                # NOTE: GitHub does NOT expose which SSH key was used to AUTHENTICATE
                # a push via the public API. Only SSH-SIGNED commits contain key info.
            })
    except GithubException as e:
        print(f"  SSH keys for {account_tag}: {e.status}")
    return records


def fetch_gpg_keys(account_tag: str, token: str) -> list:
    records = []
    resp = requests.get(
        "https://api.github.com/user/gpg_keys",
        headers={"Authorization": f"token {token}", "Accept": "application/vnd.github+json"},
    )
    if resp.status_code == 200:
        for gpg in resp.json():
            records.append({
                "account":     account_tag,
                "key_id":      gpg.get('key_id'),
                "name":        gpg.get('name'),
                "emails":      [e['email'] for e in gpg.get('emails', [])],
                "can_sign":    gpg.get('can_sign', False),
                "created_at":  gpg.get('created_at'),
                "expires_at":  gpg.get('expires_at'),
                "revoked":     gpg.get('revoked', False),
            })
    else:
        print(f"  GPG keys for {account_tag}: HTTP {resp.status_code}")
    return records


def fetch_deploy_keys(repos_csv: Path, token: str, account_tag: str) -> list:
    if not repos_csv.exists():
        return []
    df = pd.read_csv(repos_csv)
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Deploy keys [{account_tag}]"):
        resp = requests.get(
            f"https://api.github.com/repos/{row['repo_full_name']}/keys",
            headers={"Authorization": f"token {token}", "Accept": "application/vnd.github+json"},
        )
        if resp.status_code == 200:
            for dk in resp.json():
                records.append({
                    "account":     account_tag,
                    "repo":        row['repo_full_name'],
                    "key_id":      dk['id'],
                    "title":       dk['title'],
                    "fingerprint": compute_ssh_fingerprint(dk.get('key', '')),
                    "read_only":   dk['read_only'],
                    "created_at":  dk.get('created_at', ''),
                })
        time.sleep(0.1)
    return records


# ── Collect ──────────────────────────────────────────────────────────────────
print("Fetching SSH keys ...")
ssh_keys = (fetch_ssh_keys(user_personal, PERSONAL_TAG, GITHUB_TOKEN_PERSONAL)
          + fetch_ssh_keys(user_opa,      OPA_TAG,      GITHUB_TOKEN_OPA))
ssh_df = pd.DataFrame(ssh_keys)
ssh_df.to_csv(EXT_DIR / "ssh_keys.csv", index=False)

print("Fetching GPG keys ...")
gpg_keys = (fetch_gpg_keys(PERSONAL_TAG, GITHUB_TOKEN_PERSONAL)
          + fetch_gpg_keys(OPA_TAG,      GITHUB_TOKEN_OPA))
gpg_df = pd.DataFrame(gpg_keys)
gpg_df.to_csv(EXT_DIR / "gpg_keys.csv", index=False)

print("Fetching deploy keys ...")
deploy_keys = (fetch_deploy_keys(OUTPUT_DIR / "repos.csv", GITHUB_TOKEN_PERSONAL, PERSONAL_TAG))
deploy_df = pd.DataFrame(deploy_keys)
deploy_df.to_csv(EXT_DIR / "deploy_keys.csv", index=False)

print("\n" + "=" * 70)
print("SSH KEYS REGISTERED ON GITHUB (both accounts)")
print("=" * 70)
if not ssh_df.empty:
    print(ssh_df[['account', 'title', 'key_type', 'fingerprint', 'created_at', 'read_only']].to_string(index=False))
else:
    print("  None found (check token scopes — needs read:user)")

print("\n" + "=" * 70)
print("GPG KEYS (used for commit signing)")
print("=" * 70)
if not gpg_df.empty:
    print(gpg_df[['account', 'key_id', 'name', 'emails', 'can_sign', 'created_at', 'revoked']].to_string(index=False))
else:
    print("  None found.")

if not deploy_df.empty:
    print("\n" + "=" * 70)
    print("DEPLOY KEYS (per-repo)")
    print("=" * 70)
    print(deploy_df[['account', 'repo', 'title', 'fingerprint', 'read_only', 'created_at']].to_string(index=False))


NameError: name 'Path' is not defined

## Section C — OAuth Apps & GitHub App Integrations

Third-party apps with write access can push commits on your behalf — a common spam-flag trigger.
The GitHub API only exposes *GitHub App* installations; OAuth apps must be checked manually.

In [ ]:
def fetch_github_app_installations(token: str, account_tag: str) -> list:
    resp = requests.get(
        "https://api.github.com/user/installations",
        headers={"Authorization": f"token {token}", "Accept": "application/vnd.github+json"},
    )
    records = []
    if resp.status_code == 200:
        for inst in resp.json().get('installations', []):
            records.append({
                "account":               account_tag,
                "app_slug":              inst.get('app_slug'),
                "app_id":                inst.get('app_id'),
                "target_type":           inst.get('target_type'),
                "repository_selection":  inst.get('repository_selection'),
                "permissions":           json.dumps(inst.get('permissions', {})),
                "events":                ",".join(inst.get('events', [])),
                "created_at":            inst.get('created_at'),
                "updated_at":            inst.get('updated_at'),
                "suspended_at":          inst.get('suspended_at'),  # non-null = currently suspended
            })
    else:
        print(f"  Installations [{account_tag}]: HTTP {resp.status_code}")
    return records


apps_p = fetch_github_app_installations(GITHUB_TOKEN_PERSONAL, PERSONAL_TAG)
apps_o = fetch_github_app_installations(GITHUB_TOKEN_OPA,      OPA_TAG)
apps_df = pd.DataFrame(apps_p + apps_o)
apps_df.to_csv(EXT_DIR / "github_app_installations.csv", index=False)

print("GitHub App Installations (write-access to your repos):")
if not apps_df.empty:
    print(apps_df[['account', 'app_slug', 'repository_selection', 'created_at', 'suspended_at']].to_string(index=False))
else:
    print("  No GitHub App installations found.")

print()
print("OAuth apps (classic) — must be reviewed manually:")
print(f"  Personal : https://github.com/settings/applications")
print(f"  Org      : https://github.com/organizations/{ORG_LOGIN}/settings/oauth_application_policy")
print()
print("Look for apps with 'repo' or 'public_repo' scope — they can push commits.")


## Section D — Enhanced Commit Forensics

Adds **timezone offset**, **signing key info**, and **session clustering** to every commit.
The timezone offset is the most reliable machine-distinguishing signal — it's embedded in
the git object at commit time and cannot be changed without re-writing history.

In [ ]:
COMMITS_CSV = OUTPUT_DIR / "commits" / "all_commits.csv"
assert COMMITS_CSV.exists(), f"Run github_audit.ipynb first — {COMMITS_CSV} not found."

commits_df = pd.read_csv(COMMITS_CSV)
commits_df['git_author_date'] = pd.to_datetime(commits_df['git_author_date'], utc=True, errors='coerce')
print(f"Loaded {len(commits_df)} commits from main notebook.")


def extract_tz_offset(raw_date_str: str) -> str:
    """
    Pull the UTC offset from an ISO-8601 date string.
    '2024-01-15T10:30:00+02:00' -> '+02:00'   (machine in UTC+2)
    '2024-01-15T08:30:00Z'      -> '+00:00'   (machine in UTC or configured UTC)
    This offset is set by the machine's timezone at commit time.
    """
    if not isinstance(raw_date_str, str):
        return 'unknown'
    match = re.search(r'([+-]\d{2}:\d{2})$', raw_date_str.strip())
    if match:
        return match.group(1)
    if raw_date_str.strip().endswith('Z'):
        return '+00:00'
    return 'unknown'


def fetch_commit_forensics(repo_name: str, sha: str) -> dict:
    """
    Fetch raw git commit object for timezone + signing details.
    Uses personal token first, falls back to org token.
    """
    for token in [GITHUB_TOKEN_PERSONAL, GITHUB_TOKEN_OPA]:
        try:
            resp = requests.get(
                f"https://api.github.com/repos/{repo_name}/git/commits/{sha}",
                headers={"Authorization": f"token {token}",
                         "Accept": "application/vnd.github+json"},
                timeout=10,
            )
            if resp.status_code == 200:
                data = resp.json()
                author    = data.get('author',    {})
                committer = data.get('committer', {})
                verif     = data.get('verification', {})

                # For SSH-signed commits the signature starts with "-----BEGIN SSH SIGNATURE-----"
                # For GPG-signed commits it starts with "-----BEGIN PGP SIGNATURE-----"
                sig = verif.get('signature') or ''
                signing_method = (
                    'ssh' if 'BEGIN SSH SIGNATURE' in sig else
                    'gpg' if 'BEGIN PGP SIGNATURE' in sig else
                    'none'
                )
                return {
                    "sha":                 sha,
                    "author_tz_offset":    extract_tz_offset(author.get('date', '')),
                    "committer_tz_offset": extract_tz_offset(committer.get('date', '')),
                    "author_raw_date":     author.get('date', ''),
                    "committer_raw_date":  committer.get('date', ''),
                    "verified":            verif.get('verified', False),
                    "verification_reason": verif.get('reason', ''),
                    "signing_method":      signing_method,
                    # First 300 chars of payload — for GPG contains key ID, for SSH contains pubkey
                    "signing_payload_head": (verif.get('payload') or '')[:300],
                }
        except Exception:
            pass
    return {"sha": sha, "author_tz_offset": "fetch_failed"}


# ── Fetch with checkpoint ──────────────────────────────────────────────────
ENRICHED_CSV = EXT_DIR / "commits_enriched.csv"

if ENRICHED_CSV.exists():
    enriched_existing = pd.read_csv(ENRICHED_CSV)
    done_shas = set(enriched_existing['sha'].dropna().tolist())
    print(f"Checkpoint: {len(done_shas)} commits already enriched.")
else:
    enriched_existing = pd.DataFrame()
    done_shas = set()

pending = commits_df[~commits_df['sha'].isin(done_shas)]
print(f"Commits to enrich: {len(pending)}")

enriched_records = enriched_existing.to_dict('records') if not enriched_existing.empty else []

for i, (_, row) in enumerate(tqdm(pending.iterrows(), total=len(pending), desc="Enriching commits")):
    result = fetch_commit_forensics(row['repo'], row['sha'])
    enriched_records.append(result)
    if (i + 1) % 200 == 0:
        pd.DataFrame(enriched_records).to_csv(ENRICHED_CSV, index=False)
    time.sleep(0.05)

enriched_df = pd.DataFrame(enriched_records)
enriched_df.to_csv(ENRICHED_CSV, index=False)

# Merge back
commits_enriched = commits_df.merge(enriched_df, on='sha', how='left', suffixes=('', '_new'))
# Resolve column conflicts
for col in ['verified', 'verification_reason']:
    col_new = col + '_new'
    if col_new in commits_enriched.columns:
        commits_enriched[col] = commits_enriched[col_new].combine_first(commits_enriched[col])
        commits_enriched.drop(columns=[col_new], inplace=True)

commits_enriched.to_csv(EXT_DIR / "all_commits_full.csv", index=False)

print(f"\n✅ Enriched: {len(commits_enriched)} rows")
print("\nTimezone offsets (author):")
print(commits_enriched['author_tz_offset'].value_counts().to_string())
print("\nSigning methods:")
print(commits_enriched.get('signing_method', pd.Series()).value_counts().to_string())


## Section E — Machine Attribution Heuristics

Combines signals into a **machine fingerprint** per commit.
Same fingerprint = very likely the same machine.

In [ ]:
if 'commits_enriched' not in dir():
    commits_enriched = pd.read_csv(EXT_DIR / "all_commits_full.csv")
    commits_enriched['git_author_date'] = pd.to_datetime(
        commits_enriched['git_author_date'], utc=True, errors='coerce')

df = commits_enriched.copy()

# ── Signals ───────────────────────────────────────────────────────────────
# Signal 1: git identity (name + email from ~/.gitconfig on the machine)
df['git_identity'] = (df['git_author_name'].fillna('?')
                      + ' <' + df['git_author_email'].fillna('?') + '>')

# Signal 2: timezone offset embedded in commit timestamp
df['tz'] = df.get('author_tz_offset', 'unknown').fillna('unknown')

# Signal 3: account context (which GitHub account owns the repo)
repos_df = pd.read_csv(OUTPUT_DIR / "repos.csv") if (OUTPUT_DIR / "repos.csv").exists() else pd.DataFrame()
personal_repos = set(repos_df['repo_full_name'].tolist()) if not repos_df.empty else set()
df['account_context'] = df['repo'].apply(
    lambda r: PERSONAL_TAG if r in personal_repos else OPA_TAG)

# Signal 4: signing method
df['signing'] = df.get('signing_method', 'none').fillna('none')

# ── Composite fingerprint ─────────────────────────────────────────────────
df['machine_fingerprint'] = (df['git_identity'] + ' | tz='
                             + df['tz'] + ' | acct=' + df['account_context']
                             + ' | sign=' + df['signing'])

fps = {fp: f"Machine-{i+1:02d}"
       for i, fp in enumerate(df['machine_fingerprint'].unique())}
df['machine_label'] = df['machine_fingerprint'].map(fps)

# ── Report ────────────────────────────────────────────────────────────────
print("=" * 72)
print("MACHINE ATTRIBUTION — COMMIT CLUSTERS")
print("=" * 72)
print("Confidence: HIGH (all 4 signals agree) | MEDIUM (3) | LOW (2 or fewer)")
print()

summary = df.groupby('machine_label').agg(
    commits     = ('sha',              'count'),
    first       = ('git_author_date',  'min'),
    last        = ('git_author_date',  'max'),
    repos       = ('repo',             lambda x: x.nunique()),
    signed      = ('signing',          lambda x: (x != 'none').sum()),
    identity    = ('git_identity',     'first'),
    tz          = ('tz',               'first'),
    acct        = ('account_context',  'first'),
    sign_method = ('signing',          'first'),
).reset_index()

for _, r in summary.sort_values('commits', ascending=False).iterrows():
    print(f"  {r['machine_label']}")
    print(f"    Identity      : {r['identity']}")
    print(f"    Timezone      : {r['tz']}")
    print(f"    Account ctx   : {r['acct']}")
    print(f"    Signing       : {r['sign_method']}")
    print(f"    Commits       : {r['commits']}  (signed: {r['signed']})")
    print(f"    Repos touched : {r['repos']}")
    print(f"    Active        : {str(r['first'])[:10]} → {str(r['last'])[:10]}")
    print()

df.to_csv(EXT_DIR / "all_commits_attributed.csv", index=False)
save_json(fps, EXT_DIR / "machine_fingerprint_map.json")

# ── Fill-in template for court exhibit ────────────────────────────────────
template = {}
for label in fps.values():
    template[label] = {
        "ssh_key_title":       "FILL IN — e.g. 'macbook-home'",
        "ssh_key_fingerprint": "FILL IN — from: ssh-keygen -l -E sha256 -f ~/.ssh/KEY.pub",
        "physical_machine":    "FILL IN — e.g. 'MacBook Pro 16-inch 2021, home office'",
        "operating_system":    "FILL IN — e.g. 'macOS 14.4'",
        "git_version":         "FILL IN — from: git --version",
        "git_config_name":     "FILL IN — from: git config user.name",
        "git_config_email":    "FILL IN — from: git config user.email",
        "git_config_signingkey": "FILL IN — from: git config user.signingkey (if set)",
        "notes":               "",
    }
save_json(template, EXT_DIR / "machine_key_map_FILL_IN.json")
print(f"✅ Fill-in template: {EXT_DIR / 'machine_key_map_FILL_IN.json'}")
print("   Complete this file — it becomes a court exhibit linking commit clusters to physical machines.")


In [ ]:
# ── Timeline chart: commits coloured by machine ───────────────────────────
if 'df' not in dir() or df.empty:
    df = pd.read_csv(EXT_DIR / "all_commits_attributed.csv")
    df['git_author_date'] = pd.to_datetime(df['git_author_date'], utc=True, errors='coerce')

machines = df['machine_label'].dropna().unique()
cmap = plt.cm.get_cmap('tab10', max(len(machines), 1))
color_map = {m: cmap(i) for i, m in enumerate(sorted(machines))}

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

# Top: stacked bar by week
ax = axes[0]
pivot = (df.dropna(subset=['git_author_date', 'machine_label'])
           .set_index('git_author_date')
           .groupby([pd.Grouper(freq='W'), 'machine_label'])
           .size()
           .unstack(fill_value=0))
if not pivot.empty:
    pivot.plot(kind='bar', stacked=True, ax=ax,
               color=[color_map.get(c, 'grey') for c in pivot.columns],
               width=0.9, legend=True)
ax.set_title('Weekly Commits — Coloured by Inferred Machine')
ax.set_ylabel('Commits')
ax.tick_params(axis='x', labelbottom=False)
ax.legend(title='Machine', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# Bottom: dot plot
ax2 = axes[1]
for machine, grp in df.dropna(subset=['git_author_date', 'machine_label']).groupby('machine_label'):
    ax2.scatter(grp['git_author_date'], [machine] * len(grp),
                c=[color_map.get(machine, 'grey')], alpha=0.4, s=12)
ax2.set_title('Commit Dots per Inferred Machine (each dot = 1 commit)')
ax2.set_xlabel('Date')

plt.tight_layout()
chart_path = EXT_DIR / "machine_attribution_timeline.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Chart saved: {chart_path}")


## Section F — Cross-Account Merged Timeline

Collects commits from `Opa-org` repos using the org token and merges with personal commits.
Also flags commits where the same SHA appears in both contexts (you committed to an org repo).

In [ ]:
OPA_REPOS_CSV   = OUTPUT_DIR / "opa_repos.csv"
OPA_COMMITS_CSV = OUTPUT_DIR / "commits" / "opa_commits.csv"
OPA_DONE_JSON   = OUTPUT_DIR / "commits" / "opa_done_repos.json"


def collect_org_repos(org_entity, is_org: bool) -> pd.DataFrame:
    records = []
    repo_iter = org_entity.get_repos(type='all') if is_org else org_entity.get_repos(type='all')
    for r in tqdm(repo_iter, desc="Opa repos"):
        rate_limit_wait(g_opa)
        records.append({
            "account":        OPA_TAG,
            "repo_full_name": r.full_name,
            "repo_name":      r.name,
            "private":        r.private,
            "fork":           r.fork,
            "archived":       r.archived,
            "created_at":     ts(r.created_at),
            "pushed_at":      ts(r.pushed_at),
            "primary_language": r.language,
            "clone_url":      r.clone_url,
        })
    return pd.DataFrame(records)


# Load or fetch org repos
if OPA_REPOS_CSV.exists():
    opa_repos_df = pd.read_csv(OPA_REPOS_CSV)
    print(f"Checkpoint: {len(opa_repos_df)} Opa repos.")
else:
    print("Fetching Opa-org repositories ...")
    opa_repos_df = collect_org_repos(org_opa, opa_is_org)
    opa_repos_df.to_csv(OPA_REPOS_CSV, index=False)
    print(f"  Saved {len(opa_repos_df)} repos.")


# Load or fetch org commits
if OPA_COMMITS_CSV.exists():
    opa_commits_df = pd.read_csv(OPA_COMMITS_CSV)
    opa_done = load_json(OPA_DONE_JSON)
    print(f"Checkpoint: {len(opa_commits_df)} Opa commits.")
else:
    opa_commits_df = pd.DataFrame()
    opa_done = []

opa_records = opa_commits_df.to_dict('records') if not opa_commits_df.empty else []
opa_pending = [r for r in opa_repos_df['repo_full_name'].tolist() if r not in opa_done]
print(f"Opa repos remaining: {len(opa_pending)}")

# Collect commits from org repos by both account authors
for repo_name in tqdm(opa_pending, desc="Opa commits"):
    rate_limit_wait(g_opa)
    try:
        repo = g_opa.get_repo(repo_name)
        for author_login in [user_opa.login, user_personal.login]:
            try:
                for commit in repo.get_commits(author=author_login):
                    rate_limit_wait(g_opa)
                    c = commit.commit
                    try:
                        s = commit.stats
                        add, dl = s.additions, s.deletions
                    except Exception:
                        add = dl = None
                    opa_records.append({
                        "account":           OPA_TAG,
                        "github_author":     author_login,
                        "repo":              repo_name,
                        "sha":               commit.sha,
                        "short_sha":         commit.sha[:7],
                        "git_author_name":   c.author.name  if c.author else None,
                        "git_author_email":  c.author.email if c.author else None,
                        "git_author_date":   ts(c.author.date) if c.author else None,
                        "message":           c.message,
                        "message_first_line": c.message.split('\n')[0].strip(),
                        "additions":         add,
                        "deletions":         dl,
                        "files_changed":     len(commit.files),
                        "is_merge_commit":   len(commit.parents) > 1,
                        "html_url":          commit.html_url,
                    })
            except GithubException:
                pass
    except GithubException as e:
        print(f"  {repo_name}: {e.status}")
    opa_done.append(repo_name)
    pd.DataFrame(opa_records).to_csv(OPA_COMMITS_CSV, index=False)
    save_json(opa_done, OPA_DONE_JSON)

opa_commits_df = pd.DataFrame(opa_records)
print(f"\n✅ Opa commits total: {len(opa_commits_df)}")

# ── Merge ──────────────────────────────────────────────────────────────────
personal_df = pd.read_csv(OUTPUT_DIR / "commits" / "all_commits.csv")
personal_df['account'] = PERSONAL_TAG
if 'account' not in opa_commits_df.columns:
    opa_commits_df['account'] = OPA_TAG

merged = pd.concat([personal_df, opa_commits_df], ignore_index=True)
merged['git_author_date'] = pd.to_datetime(merged['git_author_date'], utc=True, errors='coerce')
merged = merged.sort_values('git_author_date')

sha_counts = merged['sha'].value_counts()
merged['cross_account'] = merged['sha'].isin(sha_counts[sha_counts > 1].index)

merged.to_csv(EXT_DIR / "merged_timeline.csv", index=False)

print("\n" + "=" * 60)
print("MERGED TIMELINE SUMMARY")
print("=" * 60)
for acct, grp in merged.groupby('account'):
    print(f"\n  [{acct}]")
    print(f"    Commits      : {len(grp)}")
    print(f"    Repos        : {grp['repo'].nunique()}")
    print(f"    Author emails: {list(grp['git_author_email'].dropna().unique())}")
    print(f"    Lines added  : {grp['additions'].sum():,.0f}")
    print(f"    Lines deleted: {grp['deletions'].sum():,.0f}")

print(f"\n  Cross-account commits (same SHA in both contexts): {merged['cross_account'].sum()}")


## Section G — Limitations Statement (Court Documentation)

Formal documentation of what the GitHub API can and cannot prove — essential for court credibility.
A clear limitations statement strengthens rather than weakens your position.

In [ ]:
from datetime import datetime, timezone

doc = f"""
TECHNICAL LIMITATIONS STATEMENT
GitHub SSH Attribution — What the API Can and Cannot Prove
Generated  : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}
Accounts   : {PERSONAL_LOGIN} (personal) / {ORG_LOGIN} (organization)
Prepared by: automated github_audit_extension.ipynb

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT IS DIRECTLY PROVABLE FROM THE GITHUB REST API
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. REGISTERED SSH KEYS (endpoint: GET /user/keys)
   - Key title (label assigned when the key was added to the account)
   - SHA256 fingerprint of the public key
   - Date the key was registered on GitHub
   - Whether the key is read-only or has push access

2. COMMIT AUTHORSHIP (permanently embedded in the git object)
   - git user.name   — set in ~/.gitconfig on the committing machine
   - git user.email  — set in ~/.gitconfig on the committing machine
   - Timestamp INCLUDING timezone offset (e.g. +02:00 = UTC+2)
   - Author vs Committer distinction (differs on amended or rebased commits)
   - Tree SHA and parent SHAs (proof of commit graph integrity)

3. COMMIT SIGNING (for GPG or SSH-signed commits)
   - Verified status and reason (as determined by GitHub)
   - GPG key ID (for PGP-signed commits)
   - SSH signing key fingerprint (recoverable for SSH-signed commits, git >=2.34)

4. ACCOUNT EVENTS (last ~90 days, GET /users/{{login}}/events)
   - Push events, issue/PR activity, repo creation, forks, stars
   - Event timestamps and associated repository

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT IS NOT AVAILABLE VIA THE PUBLIC API
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. WHICH SSH KEY authenticated a git push
   GitHub does not log or expose SSH authentication key selection via
   any public REST or GraphQL API endpoint. This information exists
   only in GitHub Enterprise audit logs (Enterprise plan required).
   Reference: https://docs.github.com/en/authentication/

2. IP ADDRESS of the pushing machine
   Not exposed via any public GitHub API.

3. MACHINE HOSTNAME
   Not embedded in git commits unless added by custom git hooks.

4. HISTORICAL OAUTH TOKEN USAGE
   Which OAuth token or PAT was used for API calls or HTTPS pushes
   is not accessible. GitHub deprecated the GET /authorizations endpoint.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FORENSIC INFERENCE METHOD
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In absence of direct SSH authentication logs, this report infers
machine origin from four compounding signals:

  Signal 1 — git user.name + git user.email (reflects local ~/.gitconfig)
  Signal 2 — Timezone offset embedded in commit timestamp
  Signal 3 — Account context (which GitHub account owns the target repo)
  Signal 4 — Signing method and key (when commit signing is enabled)

The combination is called the "machine fingerprint" and is labelled
Machine-01, Machine-02, etc. in the attributed commit table.

This is FORENSIC INFERENCE, not cryptographic proof. All claims based
on this method should be stated as:
  "The commit pattern is CONSISTENT WITH origin from machine X"
  not "This commit was PROVEN to originate from machine X."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HOW TO STRENGTHEN ATTRIBUTION LOCALLY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

On each physical machine, run:

  # Get all commit SHAs with signing key fingerprints:
  git log --format='%H %ae %ai %GF %GS' --all

  # Get fingerprint of each registered SSH key:
  ssh-keygen -l -E sha256 -f ~/.ssh/id_ed25519.pub
  ssh-keygen -l -E sha256 -f ~/.ssh/id_rsa.pub

  # Confirm git identity on this machine:
  git config user.name
  git config user.email
  git config user.signingkey

Fill in machine_key_map_FILL_IN.json with this information.
That file becomes a supporting declaration linking commit clusters
to specific physical machines.
"""

limitations_path = EXT_DIR / "LIMITATIONS_STATEMENT.txt"
with open(limitations_path, 'w', encoding='utf-8') as f:
    f.write(doc)

print(doc)
print(f"✅ Saved: {limitations_path}")


## Output Directory

```
github_audit/extension/
├── ssh_keys.csv                     ← SSH key fingerprints (both accounts)
├── gpg_keys.csv                     ← GPG signing keys
├── deploy_keys.csv                  ← Per-repo deploy keys
├── github_app_installations.csv     ← GitHub Apps with write access
├── commits_enriched.csv             ← Timezone + signing info per commit
├── all_commits_full.csv             ← Main commits + enrichment
├── all_commits_attributed.csv       ← + machine_label per commit
├── machine_fingerprint_map.json     ← Machine-01 → fingerprint string
├── machine_key_map_FILL_IN.json     ← ⬅ FILL THIS IN for the court exhibit
├── machine_attribution_timeline.png
├── merged_timeline.csv              ← skovnats + Opa-org combined
└── LIMITATIONS_STATEMENT.txt        ← Court documentation of API limits
```

### To complete machine attribution:
1. `ssh-keygen -l -E sha256 -f ~/.ssh/*.pub` — on **each** machine you own
2. `git log --format='%H %ae %ai %GF' --all` — locally, to get signing key fingerprints per commit
3. Fill in `machine_key_map_FILL_IN.json` with physical machine details
4. Cross-reference with `machine_fingerprint_map.json` to match `Machine-XX` labels to real machines